In [ ]:
# Parameters
WORKSPACE_ID = "990dbc7b-f5d1-4bc8-a929-9dfd509a5d52"
LAKEHOUSE_ID = "ec851642-fa89-42bc-aebf-2742845d36fe"
SCALE_SIZE   = "small"
SEED         = 42


## Step 1 — Define your DDL

In [ ]:
DDL = """
CREATE TABLE customer (
    customer_id INT PRIMARY KEY,
    first_name NVARCHAR(50),
    last_name NVARCHAR(50),
    email NVARCHAR(100),
    created_at DATETIME
);

CREATE TABLE order_header (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    total_amount DECIMAL(10,2),
    CONSTRAINT FK_order_customer FOREIGN KEY (customer_id) REFERENCES customer(customer_id)
);
"""
print("DDL ready")


## Step 2 — Parse DDL into Spindle schema

In [ ]:
from sqllocks_spindle.schema.parser import DDLParser

parser = DDLParser()
schema = parser.parse(DDL)
print(f"Tables: {[t.name for t in schema.tables]}")
print(f"Relationships: {len(schema.relationships)}")


## Step 3 — Generate synthetic data

In [ ]:
from sqllocks_spindle import Spindle

result = Spindle().generate(schema=schema, scale=SCALE_SIZE, seed=SEED)
errors = result.verify_integrity()
assert errors == [], f"FK integrity errors: {errors}"
for t, df in result.tables.items():
    print(f"  {t}: {len(df):,} rows, cols={list(df.columns)}")


## Step 4 — Write to Lakehouse

In [ ]:
from azure.identity import InteractiveBrowserCredential
from deltalake import write_deltalake

SOUND_BI_TENANT = "2536810f-20e1-4911-a453-4409fd96db8a"
token = InteractiveBrowserCredential(tenant_id=SOUND_BI_TENANT).get_token("https://storage.azure.com/.default").token
storage_opts = {"bearer_token": token, "use_emulator": "false"}

for table_name, df in result.tables.items():
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/custom_{table_name}"
    write_deltalake(path, df, mode="overwrite", storage_options=storage_opts, schema_mode="overwrite")
    print(f"  Written: custom_{table_name} ({len(df):,} rows)")

print("Done — your custom schema is now in OneLake as Delta tables")
